# Titanic Dataset: Outlier Detection & Treatment 
Topics:
1. Outlier Detection using the IQR Method
2. Outlier Detection using the Z-Score Method
3. Outlier Treatment using the Capping (Winsorization) Method



## Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline
sns.set_style("whitegrid")
pd.set_option("display.max_columns", None)

# Option 1: Load from seaborn's built-in dataset


In [ ]:
df.info()

## Q21. IQR-Based Outlier Detection on Fare
Using the IQR method, calculate the lower and upper bounds for `Fare`. How many passengers fall outside these bounds, and what percentage of the dataset do they represent?


In [ ]:

# Calculate Q1, Q3 and IQR
# Calculate Lower Bound and Upper Bound




print(f"Q1: {Q1:.2f}, Q3: {Q3:.2f}, IQR: {IQR:.2f}")
print(f"Lower bound: {lower_bound:.2f}, Upper bound: {upper_bound:.2f}")
print(f"Number of Fare outliers: {len(fare_outliers)}")
print(f"Percentage of dataset: {len(fare_outliers) / len(df) * 100:.2f}%")


## Q22. IQR-Based Outlier Detection on Age
Repeat the IQR-based outlier detection for `Age`. Are there fewer or more outliers compared to `Fare`? What does this tell you about the relative "spread" of the two features?


In [ ]:
# Delete recods where Age is Missing.


# Calculate Q1, Q3 and IQR
# Calculate Lower Bound and Upper Bound








print(f"Q1: {Q1_age:.2f}, Q3: {Q3_age:.2f}, IQR: {IQR_age:.2f}")
print(f"Lower bound: {lower_bound_age:.2f}, Upper bound: {upper_bound_age:.2f}")
print(f"Number of Age outliers: {len(age_outliers)}")
print(f"Percentage of non-missing Age values: {len(age_outliers) / len(age) * 100:.2f}%")

print(f"\nComparison -> Fare outliers: {len(fare_outliers)} ({len(fare_outliers)/len(df)*100:.2f}%) "
      f"vs Age outliers: {len(age_outliers)} ({len(age_outliers)/len(age)*100:.2f}%)")




## Q23. Z-Score-Based Outlier Detection on Fare
Calculate the Z-score for every value in the `Fare` column. Using a threshold of |Z| > 3, how many outliers does the Z-score method flag?


In [ ]:
from scipy import stats
# Calculate Mean and Standard Deviation of Fare




# Calulate Z-Score


# FInd Outliers based on Z-Score (|Z| > 3)



print(f"Mean Fare: {mean_fare:.2f}, Std Fare: {std_fare:.2f}")
print(f"Number of Z-score outliers (|Z| > 3): {len(z_outliers)}")


## Q24. Comparing IQR vs Z-Score Outliers on Fare
Compare the outliers flagged by the IQR method (Q21) and the Z-score method (Q23) for `Fare`. Do both methods agree on the same records? Where do they differ, and why might that happen given Fare's distribution?


In [ ]:
iqr_outlier_idx = set(fare_outliers.index)
z_outlier_idx = set(z_outliers.index)

both = iqr_outlier_idx & z_outlier_idx
only_iqr = iqr_outlier_idx - z_outlier_idx
only_z = z_outlier_idx - iqr_outlier_idx

print(f"Flagged by BOTH methods: {len(both)}")
print(f"Flagged ONLY by IQR: {len(only_iqr)}")
print(f"Flagged ONLY by Z-score: {len(only_z)}")


## Q25. Choosing the Right Method: Normality Check
The Z-score method assumes the data is approximately normally distributed. Plot the distribution of `Fare` and `Age` and discuss which feature is a better candidate for Z-score-based outlier detection versus IQR-based detection.


In [ ]:
# 1. Plot histograms (with KDE) for 'fare' and 'age' side by side
# 2. Optionally compute skewness for both: df['fare'].skew(), df['age'].skew()
# 3. Visually/numerically assess how close each is to a normal distribution

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Fare distribution
sns.histplot(df['fare'], kde=True, ax=axes[0], color='steelblue')
axes[0].set_title(f"Fare Distribution (skew = {df['fare'].skew():.2f})")

# Age distribution
sns.histplot(df['age'].dropna(), kde=True, ax=axes[1], color='darkorange')
axes[1].set_title(f"Age Distribution (skew = {df['age'].skew():.2f})")

plt.tight_layout()
plt.show()

print(f"Fare skewness: {df['fare'].skew():.2f}")
print(f"Age skewness: {df['age'].skew():.2f}")


---
# Section 6: Outlier Treatment (Capping Method)


## Q26. IQR-Based Capping (Winsorization) on Fare
Using the IQR bounds calculated in Q21, apply capping (winsorization) to the `Fare` column — replace values below the lower bound with the lower bound, and values above the upper bound with the upper bound. Compare the mean, median, and standard deviation of `Fare` before and after capping.


In [ ]:
# Create a copy of Dataframe to cap the outliers in 'fare' column




# Cap the outliers in the 'fare' column




#Check the statistics before and after capping
print("Before capping:")
print(f"  Mean: {df['fare'].mean():.2f}, Median: {df['fare'].median():.2f}, Std: {df['fare'].std():.2f}")

print("After capping:")
print(f"  Mean: {df_capped['fare'].mean():.2f}, Median: {df_capped['fare'].median():.2f}, Std: {df_capped['fare'].std():.2f}")


## Q27. Visualizing the Effect of Capping
Plot boxplots of `Fare` before and after capping, side by side. How does the visual spread of the data change, and are there still any points shown as outliers?


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 5))

# Boxplot of original 'fare'
sns.boxplot(y=df['fare'], ax=axes[0], color='steelblue')
axes[0].set_title("Original Fare")

# Boxplot of capped 'fare' (df_capped)
sns.boxplot(y=df_capped['fare'], ax=axes[1], color='seagreen')
axes[1].set_title("Capped Fare")

plt.tight_layout()
plt.show()


## Q28. Percentile-Based Capping
Apply percentile-based capping to `Fare` — cap all values below the 1st percentile and above the 99th percentile. How do the resulting bounds differ from the 1.5×IQR bounds used in Q26?


In [ ]:
# Cap the outliers in the 'fare' column using 1st and 99th percentiles
p01 = df['fare'].quantile(0.01)
p99 = df['fare'].quantile(0.99)

df_pct_capped = df.copy()
df_pct_capped['fare'] = np.where(df_pct_capped['fare'] > p99, p99,
                          np.where(df_pct_capped['fare'] < p01, p01, df_pct_capped['fare']))

print(f"1st percentile: {p01:.2f}, 99th percentile: {p99:.2f}")
print(f"IQR bounds (Q21/Q26): lower={lower_bound:.2f}, upper={upper_bound:.2f}")


## Q29. Z-Score-Based Capping on Age
Apply Z-score-based capping to `Age` — cap any value with |Z| > 3 to the corresponding boundary value (mean ± 3×std). Compare the number of values affected by this method versus the IQR capping approach.


In [ ]:
age_clean = df['age'].dropna()
# Calulate Mean and Standard Deviation of Age



# Calculate Lower and Upper Z bounds for Age


#Apply Capping
df_age_capped = df.copy()
df_age_capped['age'] = np.where(df_age_capped['age'] > upper_z_bound, upper_z_bound,
                        np.where(df_age_capped['age'] < lower_z_bound, lower_z_bound, df_age_capped['age']))

n_changed = ((age_clean > upper_z_bound) | (age_clean < lower_z_bound)).sum()

print(f"Mean Age: {mean_age:.2f}, Std Age: {std_age:.2f}")
print(f"Lower Z bound: {lower_z_bound:.2f}, Upper Z bound: {upper_z_bound:.2f}")
print(f"Number of Age values capped (Z-score method): {n_changed}")
print(f"Number of Age outliers from IQR method (Q22): {len(age_outliers)}")


## Q30. Capping vs Removal — Discussion
Capping changes the underlying distribution of a feature rather than removing data points. Discuss: in what situations would you prefer capping over removing outliers entirely, and what risk does capping introduce if applied carelessly to a legitimately skewed feature like `Fare`?
